In [0]:
%pip install dotenv openpyxl 

In [0]:
#Imports
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.functions import countDistinct
import plotly.express as px
import warnings
from dotenv import load_dotenv
import os
from pathlib import Path

warnings.simplefilter(action='ignore')
load_dotenv()

In [0]:
project_master_opcs = '/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/14_07_2026/PROJECT_MASTER_14_07_2026.xlsx'
path_prior_actions = 'https://thedocs.worldbank.org/en/doc/ed4dbe34c2cd2555d0d7b28952bddf54-0290032023/original/DPADdatabaseFY22.xlsx'

hierarchy_table = spark.table('prd_mega.sgpbpi163.0c_hierarchy_table_goat')

ind_link = '/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/14_07_2026/PROJECT_RESULT_IND_DETAIL_V2_07_14_2026(PROJECT_RESULT_IND_DETAIL_V2_07).xlsx'
path_proj_component = '/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/14_07_2026/PROJECT_COMPONENT_LIST_V3_07_14_2026(PROJECT_COMPONENT_LIST_V3_07_14).xlsx'


In [0]:
sel_projects = 'all'

# sel_hierarchy can be 'decentralization' or 'pfm' or 'asset management'
sel_hierarchy = 'pima'

#selected countries country_selected can either be 'all' or a list of countries ['Country1','Country']
country_selected = 'all'

#minimum and maximum year to be filtered
min_year = 2000
max_year = 2026

In [0]:
df_prior_actions = pd.read_excel(path_prior_actions,sheet_name='Prior Actions database')
df_project_master = pd.read_excel(project_master_opcs, header=None)
# Convert the pandas DataFrame to a Spark DataFrame and register as a temporary view
# df_project_master= spark.createDataFrame(df_project_master)
# df_project_master_spark.createOrReplaceTempView("projMaster")

In [0]:
df_project_master = pd.read_excel(project_master_opcs, header=None)

In [0]:
# df_project_master= spark.createDataFrame(df_project_master)
df_project_master

In [0]:
# df_project_master= spark.createDataFrame(df_project_master)
def remove_empty_top_cells(df, header_row_index):
    # Add a monotonically increasing row id to preserve order
    df_with_id = df.withColumn("row_id", F.monotonically_increasing_id())
    
    # Extract the header row (as a Row object) based on the supplied index
    header_row = (
        df_with_id
        .orderBy("row_id")
        .limit(header_row_index + 1)
        .collect()[header_row_index]
    )
    
    # Convert header values to strings; replace missing values with generic column names
    new_columns = [
        str(col_val) if col_val is not None else f"_c{idx}"
        for idx, col_val in enumerate(header_row[:-1])  # exclude the added row_id column
    ]
    
    # Remove the header rows from the data and drop the temporary row_id column
    df_clean = (
        df_with_id
        .orderBy("row_id")
        .filter(F.col("row_id") > header_row_index)
        .drop("row_id")
    )
    
    # Rename columns using the extracted header values
    df_clean = df_clean.toDF(*new_columns)
    
    return df_clean

In [0]:
df_project_master = remove_empty_top_cells(df_project_master, 4)
display(df_project_master)
df_project_master_subset = df_project_master.toPandas() 

In [0]:
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_ID']!='P169212']

df_project_master_subset = df_project_master_subset[df_project_master_subset['LNDNG_INSTR_LONG_NAME'].isin(['Development Policy Lending','Investment Project Financing','Program-for-Results Financing'])]

df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_STAT_NAME'].isin(['Closed','Active'])]

if (country_selected=='all'):
  df_project_master_subset = df_project_master_subset
else:
  df_project_master_subset = df_project_master_subset[df_project_master_subset['CNTRY_SHORT_NAME'].isin(country_selected)] 

# min_year, max_year = 2018, 2025
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(str)!='None']
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)>=min_year]
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)<=max_year]

In [0]:
df_project_master_subset = df_project_master_subset[['PROJ_ID','PROJ_DISPLAY_NAME','PROJ_APPRVL_FY','PROJ_DEV_OBJECTIVE_DESC',
                                                    'RGN_NAME','PROJ_STAT_NAME','CNTRY_SHORT_NAME',
                                                    'PROD_LINE_NAME','LNDNG_INSTR_TYPE_CODE','LNDNG_INSTR_LONG_NAME','CMT_AMT',
                                                    'PARENT_PROJ_ID','TEAM_LEAD_FULL_NAME','LEAD_GP_NAME','PROJ_SHORT_NAME',
                                                    'PROJ_LGL_NAME','PROD_LINE_TYPE_NAME','DLI_IND']]
df_project_master_subset['PROJ_ID'].nunique()

In [0]:
df_pdo_search = df_project_master_subset[['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC', 'PROJ_APPRVL_FY', 'CNTRY_SHORT_NAME', 'PROJ_DISPLAY_NAME']]

# def check_pfm_categories(x,category):
#   if(x!=None):
#     category_words = hierarchy[category]
#     res = any(ele in x for ele in category_words)  
#     if(res==True):
#       return 'Yes'
#     else:
#       return None
#   else:
#     return None

# for each_group in hierarchy:
#   df_pdo_search[each_group] = df_pdo_search['PROJ_DEV_OBJECTIVE_DESC'].apply(check_pfm_categories,category=each_group)

df_pdo_search_selected = df_pdo_search.set_index(['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']).dropna(how='all').reset_index()

display(df_pdo_search_selected.astype(str))

In [0]:
merged_df = pd.merge(df_pdo_search_selected, df_project_master_subset,
                     on=['PROJ_ID', 'PROJ_DEV_OBJECTIVE_DESC']
                    )

In [0]:
df_prior_actions[df_prior_actions['Project ID'].isin(list_projects)]['Project ID'].nunique()

In [0]:
# Read the indicator Excel file with all columns as strings to avoid type mismatches
df_results_indicators = pd.read_excel(ind_link, header=None, dtype=str)

# Convert the pandas DataFrame to a Spark DataFrame
df_results_indicators = spark.createDataFrame(df_results_indicators)
df_results_indicators = remove_empty_top_cells(df_results_indicators, 4)

df_results_indicators = df_results_indicators.toPandas()

# PDO and Intermediate Results Indicator
df_results_indicators = df_results_indicators[['PROJ_ID','IND_TYPE_NAME','IND_NAME','BASELINE_VAL_TEXT','TGT_VAL_TEXT','UOM_NAME']]
df_results_indicators = df_results_indicators[df_results_indicators['PROJ_ID'].isin(df_project_master_subset['PROJ_ID'].unique())]

In [0]:
df_results_indicators

In [0]:
df_results_indicators.groupby('IND_TYPE_NAME')['PROJ_ID'].nunique()
df_results_indicators['PROJ_ID'].nunique()

In [0]:
df_prior_actions_selected = df_prior_actions[df_prior_actions['Project ID'].isin(list_projects)][['Project ID','TEXT']]

# def check_pfm_categories(x,category):
#   if(x!=None):
#     category_words = hierarchy[category]
#     res = any(ele in x for ele in category_words)  
#     if(res==True):
#       return 'Yes'
#     else:
#       return None
#   else:
#     return None

# for each_group in hierarchy:
#   df_prior_actions_selected[each_group] = df_prior_actions_selected['TEXT'].apply(check_pfm_categories,category=each_group)

# df_prior_actions_selected = df_prior_actions_selected.set_index(['Project ID','TEXT']).dropna(how='all').reset_index()

# display(df_prior_actions_selected.astype(str))

In [0]:
df_proj_component = pd.read_excel(path_proj_component, header=None, dtype=str)
df_proj_component = spark.createDataFrame(df_proj_component)
df_proj_component = remove_empty_top_cells(df_proj_component, 4)
df_proj_component = df_proj_component.toPandas()

In [0]:
# Aggregate indicator names per project (unique, comma‑separated)
df_result_cleaned = df_project_master_subset
indicators_df = (
    df_results_indicators
    .groupby("PROJ_ID")["IND_NAME"]
    .apply(lambda s: ", ".join(s.dropna().unique()))
    .reset_index()
    .rename(columns={"IND_NAME": "Indicators"})
)

# Merge the aggregated indicators into the cleaned result set
df_result_cleaned = df_result_cleaned.merge(
    indicators_df,
    on="PROJ_ID",
    how="left"
)
display(df_result_cleaned)

In [0]:
prior_actions_df = (
    df_prior_actions
    .groupby("Project ID")["TEXT"]
    .apply(lambda s: ", ".join(s.dropna().unique()))
    .reset_index()
    .rename(columns={"Project ID":"PROJ_ID","TEXT": "PriorActions"})
)
prior_actions_df

In [0]:
df_result_cleaned = df_result_cleaned.merge(
    prior_actions_df,
    on="PROJ_ID",
    how="left"
)
display(df_result_cleaned)

In [0]:
df_proj_component = df_proj_component[['PROJ_ID','CMPNT_NAME']]
component_df = (
    df_proj_component
    .groupby("PROJ_ID")["CMPNT_NAME"]
    .apply(lambda s: ", ".join(s.dropna().unique()))
    .reset_index()
    .rename(columns={"CMPNT_NAME": "Components"})
)
display(component_df)

In [0]:
df_result_cleaned = df_result_cleaned.merge(
    component_df,
    on="PROJ_ID",
    how="left"
)
display(df_result_cleaned)

In [0]:
df_result_cleaned.shape

In [0]:
dli = spark.table('prd_mega.sgpbpi163.0_goat_dli_master')

In [0]:
dli = dli.toPandas()
df_result_cleaned = df_result_cleaned.merge(
    dli,
    on="PROJ_ID",
    how="left"
)
display(df_result_cleaned)
df_result_cleaned

In [0]:
# Convert the pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(df_result_cleaned)
# Write the Spark DataFrame as a Delta table in the target schema
spark_df.write.format("delta").mode("overwrite").saveAsTable("prd_mega.sgpbpi163.2a_goat_master")